# Brief Intro to Likelihood Functions for Parameter Estimation and Hypothesis Testing

Written by Jeff Hyde, jhyde1@swarthmore.edu

The purpose of this notebook is to give a very quick motivation and introduction to likelihood functions and hypothesis testing using the likelihood ratio test, before applying these in a different notebook to learn about astrophysical neutrinos.

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Evaluate data relative to a distribution:

The probability density function (pdf) tells us how likely each possible result is, given the parameters chosen. For example, if we know (or suspect) that in some experiment the probability of each particle having energy E is normally distributed about some $E_0$ with the spread characterized by $\sigma$, then the pdf is f(E) = $\exp(-(E-E_0)^2 / 2 \sigma^2) / \sqrt(2 \pi \sigma^2)$.

In practical cases, we often know the data and don't know the distribution. (Or we may know or suspect the functional form of the distribution, but not know the values of parameters.) For now we will assume that we know, somehow, that it is a normal distribution but we don't know the parameters ($E_0$ and $\sigma$).

Below, we will consider a hypothetical data set of three particle energy measurements, and think about how to estimate parameters...

In [89]:
# To evaluate the value of the pdf f(E), we can either write our own function to do this, or use Python's existing function.
# In practical situations (such as the neutrino data analysis we'll do next) the pdf will not be a simple analytic form, and 
# in fact a lot of the effort of interpreting experimental data goes into determining the correct pdf.

# Function to return value of pdf, given observed energy E_exp and parameters E_0 and \sigma of the distribution:
def gaussian_pdf(E_exp,E_0,sigma):
    normalization = np.sqrt(2.*np.pi*sigma**2)
    func = np.exp(-(E_exp - E_0)**2 / (2 * sigma**2))
    return func/normalization

# Or we can use a pre-built function for this purpose
from scipy import stats
from scipy.stats import norm
# The function "norm.pdf" takes arguments in the same order as the function I wrote above: norm.pdf(point,E0,sigma)

# Now if we want to know the value of the pdf for E = 3.0 GeV when E_0 = 4.0 GeV and sigma = 1.0 GeV:
print('My function gives:',gaussian_pdf(3.0,4.0,1.0))
print('Python normal dist:',norm.pdf(3.0,4.0,1.0))

Now, suppose we measure three particles to have energies 3.0 GeV, 2.1 GeV, and 4.5 GeV. Given the same parameters above ($E_0 = 4.0$ GeV, $\sigma = 1.0$ GeV), what was the probability of observing these three values? The probability of getting a combination of three results is the probability of each multiplied together (we assume the trials are independent of each other):

P(these three data points) = P(observed particle 1) x P(observed particle 2) x P(observed particle 3)

Below we evaluate these numerically:

In [90]:
# Probability of having observed this data set, given the above parameters:
data_set_3particle = [3.0,2.1,4.5]
probabilities = []
for observed_energy in data_set_3particle:
    prob = gaussian_pdf(observed_energy,4.0,1.0)
    probabilities.append(prob)

total_prob = probabilities[0]*probabilities[1]*probabilities[2]


print('Total probability ',total_prob)
print('Log of total prob',np.log(total_prob))

{\bf Question for you:} By modifying what was done above, find a combination of parameters ($E_0$ and $\sigma$) for which the observed data points have a higher probability. What about a lower probability?

In [ ]:
# Put your answer to the above question here.





# The Likelihood Function

This motivates the idea of the "likelihood function" L which is essentially just the joint probability density function evaluated with respect to the observed data. There is an important conceptual shift, though: we think of the data as fixed (we already observed it) and the model parameters as variables. The method of maximum likelihood is a method where we vary the relevant parameters and compute the value of the likelihood function L. The parameters which give the greatest value of L are those we choose as the "estimated" or "best fit" parameters.

More formally, if our data set consists of $N$ values $\{ x_i \}$, each being described by a probability density function $f(x|\theta)$ (where $\theta$ is a parameter such as $\sigma$ in the above example), then the likelihood function is defined as
$L = \prod_{i=1}^N f(x_i|\theta).$

Finally, since this product is typically a very small number even for the best-fit parameters, the quantity we more frequently deal with is the logarithm of the likelihood function. The log likelihood is then
$\log(L) = \sum_{i=1}^N \log(f(x_i|\theta)).$

One last point of formalism is that when there are more parameters, some authors will describe these as something like $\{\theta_i\}$ or $\vec{\theta}$, where the ``vector components'' would just be the individual parameters.


In [85]:
# Example: Maximum likelihood estimate for our three data point gaussian example above.

# Function to evaluate value of log likelihood given data set and one choice of parameters E_0, sigma.
def logL_gaussian_example(data,E0,sigma):
    logL = 0
    for data_point in data:
        logL += np.log(gaussian_pdf(data_point,E0,sigma))
    return logL

# Search parameter space with varying E0 and fixed \sigma = 1.0 GeV, plot result.
def logL_vary_E0(data,Emin,Emax):
    max_LLH = -999
    best_E0 = -999
    E0_vals = np.linspace(Emin,Emax,100) # Range of possible parameter values E0 to try.
    LLH_vals = []
    for i in range(0,len(E0_vals)):
        E0_val = E0_vals[i]
        this_LLH = logL_gaussian_example(data,E0_val,1.0)
        LLH_vals.append(this_LLH)
        if this_LLH > max_LLH:
            max_LLH = this_LLH
            best_E0 = E0_val
    print('Best fit E0 =',best_E0,'with LLH =',max_LLH)
    plt.plot(E0_vals,LLH_vals,'k')
    plt.xlabel(r'Central $E_0$ (GeV)')
    plt.ylabel(r'Log L')
    return

# Search parameter space with varying E0 and sigma, plot result.
# plot_t_f = True or False, say whether to plot result.
def logL_fit(data,Emin,Emax,smin,smax,plot_t_f):
    max_LLH = -999
    best_E0 = -999
    best_sigma = -999
    E0_vals = np.linspace(Emin,Emax,100) # Range of possible parameter values E0 to try.
    s_vals = np.linspace(smin,smax,100) # Range of possible parameter values sigma to try.
    LLH_vals = []
    for i in range(0,len(E0_vals)):
        E0_val = E0_vals[i]
        this_LLH_column = []
        for j in range(0,len(s_vals)):
            sigma_val = s_vals[j]
            this_LLH = logL_gaussian_example(data,E0_val,sigma_val)
            this_LLH_column.append(this_LLH)
            if this_LLH > max_LLH:
                max_LLH = this_LLH
                best_E0 = E0_val
                best_sigma = sigma_val
        LLH_vals.append(this_LLH_column)

    if plot_t_f == True:
        # Print result of fit
        print('Best fit E0 =',best_E0,', best fit sigma =',best_sigma,'with LLH =',max_LLH)

        Delta_LLH = max_LLH - LLH_vals
        ### Contour plot
        X, Y = np.meshgrid(E0_vals,s_vals)
        fig, ax = plt.subplots()
        im = ax.imshow(np.transpose(Delta_LLH), interpolation='bilinear', origin='lower', cmap=cm.viridis.reversed(), extent=(Emin,Emax,smin,smax), norm='log')
        CB = fig.colorbar(im, shrink=0.8)
        CB.ax.set_title(r'$ \ \Delta\log(L)$',fontsize=14)

        plt.plot(best_E0, best_sigma, marker='+', color='k', markersize=12)
        
        ax.set_aspect((Emax-Emin)/(smax-smin))
        #ax.clabel(ConPlot1, inline=False, fontsize=12)
        #ax.clabel(ConPlot2, inline=False, fontsize=12)
        ax.set_ylabel(r'$\sigma$ (GeV)', fontsize=14)
        ax.set_xlabel(r'Central $E_0$ (GeV)', fontsize=14)
    return max_LLH


In [88]:
# Use the above functions to compute and display the result

# First, only vary E0:
logL_vary_E0(data_set_3particle,1.0,10.0)

In [87]:
# Now search parameter space varying both E0 and sigma:

logL_fit(data_set_3particle,2,8,0.25,2.0,True)


Question: Make up your own data set, but include more than three points in it. How does the fit change? Do the plots of the log likelihood become more or less sharply peaked? Compare your invented data sets with other students.

In [ ]:
# Put your work relating to the above question here.




# Hypothesis Testing and Parameter Estimation via Likelihood Ratio Test

A situation that occurs very often in high-energy physics and astrophysics is that we have a model with some unknown parameters, and we want to decide whether this model ("test hypothesis") describes the data better than our "baseline" expectation ("null hypothesis"). And if the test hypothesis is true, we want an estimate of the values of the unknown parameters.

The likelihood ratio test is a frequently-used method for answering this question. For this test we define a test statistic $\lambda$ by:

$\lambda \equiv 2 \log(L_{\rm test} / L_{\rm null})$,

where both are evaluated against the same data set. Qualitatively, if the likelihood of the test hypothesis is greater than that of the null hypothesis, given the data, then $L_{\rm test} / L_{\rm null} > 1$ so the logarithm and therefore the test statistic $\lambda$ is positive. Alternatively, data that better fits the null hypothesis will lead to a negative $\lambda$. And of course, the "more positive" $\lambda$ is, the better the evidence for the test hypothesis.

So the strategy is to define the relevant region of the test hypothesis parameter space which we will search, evaluate the test statistic on a representative set of those parameters, and determine the maximum value of $\lambda$. If this maximum value of $\lambda$ is high enough, then we conclude that the test hypothesis fits the data better than the null hypothesis, and the parameter value which maximizes $\lambda$ is our best-fit estimate for those parameters.

Likelihood ratio tests have been used in many important cases, such as the discovery of the Higgs boson. We'll look at an example below.

# Example of Likelihood Ratio Test Inspired by Particle Physics

This example is inspired by an example given in Chapter 6 of Glen Cowan's "Statistical Data Analysis", which is a good overview of commonly-used statistical methods in particle physics.

In many cases, when we compute the differential cross section for some particle interaction process, such as $e^+e^- \rightarrow \mu^+\mu^-$, the angular distribution has the form:

$f(x|\alpha,\beta) = (1 + \alpha x + \beta x^2) / (2 + 2\beta / 3)$,

where $x = \cos(\theta)$ and the parameters $\alpha$ and $\beta$ will vary between 0 and 1. We will take our null hypothesis to be the Standard Model case where $\alpha = 0$ and $\beta = 1$, and our test hypothesis will be $\alpha \neq 0$.

Working in groups, use this model and the functions I've already written below (along with anything else you would like to add or modify!) to:
- Create a simulated data set, i.e. list of values of x that may be measured in some experiment. These may follow the null hypothesis or the test hypothesis. Trade data sets with another group, and don't tell each other (yet) which parameters were used to create it.
- Use a likelihood ratio test to evaluate the test statistic for the simulated data set. Do you think the null or test hypothesis is more likely? If the test hypothesis, which parameter values created the data set?

After this, we will discuss your results, and in particular we will think about the question of how large is "large enough" for $\lambda$ to mean that we reject the null hypothesis.

In [119]:
# Evaluate pdf, given data (x = cos(theta)) and parameters alpha and beta.
def evaluate_pdf(x,a,b):
    numerator = 1. + a*x + b*x*x
    denominator = 2. + 2. * b / 3.
    return numerator/denominator

# Produce one data point (x value) based on the distribution, given parameters alpha and beta
def get_x_value(a,b):
    # This function gives an example of a simple method to choose a value from a distribution, starting with a random
    # variable from a uniform distribution. There are many other ways to do this, some much more efficient! This is basically
    # equivalent to drawing a plot of the pdf and then throwing darts at it, and only keeping the ones which land underneath
    # the curve. Therefore, x values where the pdf is larger are more likely to be selected.
    
    finished = False
    while finished == False: # i.e. try the following over and over until we get the result we want.
        # Get a random x value between -1 and 1. The uniform distribution returns a variable between 0 and 1,
        # so we multiply by 2 (range is now 0 to 2) then subtract 1 (range is now -1 to 1, which we want).
        try_x = np.random.random() * 2 - 1
        # Get a random pdf value
        try_pdf = np.random.random()
        
        # Now evaluate the pdf of trial x, and see if it is more or less than the "try_pdf" value.
        # i.e. for this value of x did the dart land above or below the curve?
        this_pdf_value = evaluate_pdf(try_x,a,b)
        if this_pdf_value > try_pdf: # If the dart lands below the curve...
            result_x = try_x
            finished = True # ... then exit the loop and ...
    return result_x # ... return that value.

# Generate a set of N_data_points values of x from distribution
def get_data_set(N_data_points,a,b):
    return [get_x_value(a,b) for i in range(0,N_data_points)]
